# Hearthstone Battlegrounds Dataset Explorer

Loads all parsed game sessions and lets you explore what your final boards look like, hero performance, health trends, and more.

In [ ]:
import json
from pathlib import Path
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from hearthstone.cardxml import load_dbf

# card name lookup
db, _ = load_dbf()
_CARD_DB = {c.id: c for c in db.values()}

def card_name(card_id: str) -> str:
    c = _CARD_DB.get(card_id)
    return c.name if c else card_id

# load all sessions
DATA_DIR = Path("data")

MIN_ROUNDS = 8  # discard incomplete/quit games

games_raw = []
for f in sorted(DATA_DIR.glob("Hearthstone_*.json")):
    games_raw.extend(json.loads(f.read_text(encoding="utf-8")))

games = [g for g in games_raw if len(g.get("rounds", [])) >= MIN_ROUNDS]

print(f"Loaded {len(games)} games ({len(games_raw) - len(games)} incomplete skipped) "
      f"across {len(list(DATA_DIR.glob('Hearthstone_*.json')))} sessions")

## Dataset Overview

In [ ]:
rows = []
for g in games:
    hero   = g.get("hero") or {}
    rounds = g.get("rounds", [])
    last   = rounds[-1] if rounds else {}
    last_shopping = last.get("shopping") or {}
    last_combat   = last.get("combat")   or {}

    rows.append({
        "session":      g["session"],
        "game_index":   g["game_index"],
        "hero_card_id": hero.get("card_id", ""),
        "hero_name":    card_name(hero.get("card_id", "")),
        "placement":    g.get("placement"),
        "won":          g.get("placement") == 1,
        "num_rounds":   len(rounds),
        "anomaly":      g.get("anomaly"),
        "final_health": last_combat.get("hero_health_after", 0),
        "final_armor":  last_combat.get("hero_armor_after",  0),
        "final_tier":   last_shopping.get("tavern_tier", 0),
        "final_board":  last_combat.get("player_board", []),
    })

df = pd.DataFrame(rows)

wins   = df["won"].sum()
losses = len(df) - wins
print(f"Total games : {len(df)}")
print(f"Wins        : {wins}  ({wins/len(df)*100:.0f}%)")
print(f"Losses/other: {losses}")
print(f"Avg rounds  : {df['num_rounds'].mean():.1f}")
print()
df[["session","hero_name","won","num_rounds","final_tier"]]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].bar(["Win", "Loss/Other"], [wins, losses], color=["#4CAF50", "#F44336"])
axes[0].set_title("Win / Loss")
axes[0].set_ylabel("Games")

axes[1].hist(df["num_rounds"], bins=range(df["num_rounds"].min(), df["num_rounds"].max()+2), color="steelblue", edgecolor="white")
axes[1].set_title("Rounds per Game")
axes[1].set_xlabel("Rounds")
axes[1].set_ylabel("Games")

tier_counts = df["final_tier"].value_counts().sort_index()
axes[2].bar(tier_counts.index.astype(str), tier_counts.values, color="darkorange")
axes[2].set_title("Final Tavern Tier")
axes[2].set_xlabel("Tier")
axes[2].set_ylabel("Games")

plt.tight_layout()
plt.show()

In [ ]:
# ── Action counts ─────────────────────────────────────────────────────────────
total_actions = 0
actions_per_game = []

for g in games:
    game_actions = sum(
        len(r.get("shopping", {}).get("actions", []))
        for r in g.get("rounds", [])
    )
    total_actions += game_actions
    actions_per_game.append(game_actions)

avg_actions = total_actions / len(games) if games else 0

print(f"Total actions across all games : {total_actions:,}")
print(f"Number of games                : {len(games):,}")
print(f"Average actions per game       : {avg_actions:.1f}")

## Final Board Lineup

The board going into the **last combat** of each game, split by outcome.

In [ ]:
def board_minion_counts(game_rows, top_n=15):
    counter = Counter()
    total_games = 0
    for _, row in game_rows.iterrows():
        board = row["final_board"]
        if not board:
            continue
        total_games += 1
        for minion in board:
            name = card_name(minion["card_id"]) if minion["card_id"] else "Unknown"
            if minion.get("golden"):
                name = f"{name} â˜…"
            counter[name] += 1
    return counter.most_common(top_n), total_games

win_rows  = df[df["won"] == True]
loss_rows = df[df["won"] == False]

win_counts,  win_n  = board_minion_counts(win_rows)
loss_counts, loss_n = board_minion_counts(loss_rows)

print(f"Final boards with Wins: {win_n}, Losses: {loss_n}")

In [ ]:
def plot_board_counts(counts, title, ax, color):
    if not counts:
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(title)
        return
    names, vals = zip(*counts)
    ax.barh(list(reversed(names)), list(reversed(vals)), color=color)
    ax.set_xlabel("Times on final board")
    ax.set_title(title)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plot_board_counts(win_counts,  f"Final Board â€” Wins ({win_n} games)",  axes[0], "#4CAF50")
plot_board_counts(loss_counts, f"Final Board â€” Losses ({loss_n} games)", axes[1], "#F44336")
plt.tight_layout()
plt.show()

### Final Board Stats (Wins)

In [ ]:
def board_stats_df(game_rows):
    records = []
    for _, row in game_rows.iterrows():
        for m in row["final_board"]:
            records.append({
                "name":         card_name(m["card_id"]) if m["card_id"] else "Unknown",
                "golden":       m.get("golden", False),
                "attack":       m.get("attack", 0),
                "health":       m.get("health", 0),
                "tier":         m.get("tier", 0),
                "divine_shield":m.get("divine_shield", False),
                "taunt":        m.get("taunt", False),
                "reborn":       m.get("reborn", False),
                "poisonous":    m.get("poisonous", False),
                "windfury":     m.get("windfury", False),
            })
    return pd.DataFrame(records)

win_board_df = board_stats_df(win_rows)

if not win_board_df.empty:
    summary = (
        win_board_df.groupby("name")
        .agg(
            count=("name", "count"),
            avg_atk=("attack", "mean"),
            avg_hp=("health", "mean"),
            avg_tier=("tier", "mean"),
            golden=("golden", "sum"),
            div_shield=("divine_shield", "sum"),
            taunt=("taunt", "sum"),
            reborn=("reborn", "sum"),
        )
        .sort_values("count", ascending=False)
        .reset_index()
    )
    summary[["avg_atk","avg_hp","avg_tier"]] = summary[["avg_atk","avg_hp","avg_tier"]].round(1)
    display(summary)
else:
    print("No winning game boards found yet.")

## Hero Performance

In [ ]:
hero_stats = (
    df.groupby("hero_name")
    .agg(
        games=("won", "count"),
        wins=("won", "sum"),
        avg_rounds=("num_rounds", "mean"),
        avg_tier=("final_tier", "mean"),
    )
    .assign(win_rate=lambda x: (x["wins"] / x["games"] * 100).round(0))
    .sort_values("games", ascending=False)
    .reset_index()
)
hero_stats[["avg_rounds","avg_tier"]] = hero_stats[["avg_rounds","avg_tier"]].round(1)
display(hero_stats)

## Health Progression Over Rounds

In [ ]:
def health_series(game):
    pts = []
    for r in game["rounds"]:
        combat = r.get("combat") or {}
        hp     = combat.get("hero_health_after", None)
        armor  = combat.get("hero_armor_after",  0)
        if hp is not None:
            pts.append((r["round"], hp + armor))
    return pts

fig, ax = plt.subplots(figsize=(12, 5))

for g in games:
    pts = health_series(g)
    if not pts:
        continue
    rounds, hps = zip(*pts)
    won   = g.get("placement") == 1
    color = "#4CAF50" if won else "#F44336"
    alpha = 0.8 if won else 0.4
    ax.plot(rounds, hps, color=color, alpha=alpha, linewidth=1.5)

from matplotlib.lines import Line2D
ax.legend(handles=[
    Line2D([0],[0], color="#4CAF50", lw=2, label="Win"),
    Line2D([0],[0], color="#F44336", lw=2, alpha=0.4, label="Loss"),
])
ax.set_xlabel("Round")
ax.set_ylabel("Health + Armor")
ax.set_title("Hero Health + Armor Per Round")
ax.axhline(0, color="black", linewidth=0.5, linestyle="--")
plt.tight_layout()
plt.show()

## Keyword Frequency on Final Boards (Wins vs Losses)

In [ ]:
keywords = ["divine_shield", "taunt", "reborn", "poisonous", "windfury", "golden"]

def keyword_rates(game_rows):
    total_minions = 0
    kw_counts = Counter()
    for _, row in game_rows.iterrows():
        for m in row["final_board"]:
            total_minions += 1
            for kw in keywords:
                if m.get(kw):
                    kw_counts[kw] += 1
    if total_minions == 0:
        return {}
    return {kw: kw_counts[kw] / total_minions * 100 for kw in keywords}

win_kw  = keyword_rates(win_rows)
loss_kw = keyword_rates(loss_rows)

if win_kw or loss_kw:
    x      = range(len(keywords))
    w_vals = [win_kw.get(k, 0)  for k in keywords]
    l_vals = [loss_kw.get(k, 0) for k in keywords]

    fig, ax = plt.subplots(figsize=(10, 4))
    width = 0.35
    ax.bar([i - width/2 for i in x], w_vals, width, label="Win",  color="#4CAF50")
    ax.bar([i + width/2 for i in x], l_vals, width, label="Loss", color="#F44336", alpha=0.7)
    ax.set_xticks(list(x))
    ax.set_xticklabels([k.replace("_", " ").title() for k in keywords])
    ax.set_ylabel("% of final board minions")
    ax.set_title("Keyword Frequency on Final Board (Win vs Loss)")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Not enough board data yet.")

---
## Neurosymbolic Layer

The symbolic layer computes deterministic mechanical quantities: multiplier detection, tribal density, aura dependency scores, and effect duration profiles. These features feed directly into the Transformer policy network.

In [ ]:
# Load card definitions and build symbolic components
import sys
sys.path.insert(0, ".")

from symbolic.board_computer import SymbolicBoardComputer, BoardFeatures
from agent.card_encoder      import CardEncoder, CARD_FEATURE_DIM, TRIBE_LIST, TRIGGER_TYPES, DURATION_TYPES
import numpy as np

with open("bg_card_definitions.json") as _f:
    _defs_raw = json.load(_f)
CARD_DEFS = _defs_raw['cards']

board_computer = SymbolicBoardComputer(CARD_DEFS)
card_encoder   = CardEncoder(CARD_DEFS)

# Reverse lookup: lower-cased name → card_def
_NAME_TO_DEF = {v['name'].lower(): v for v in CARD_DEFS.values()}

def enrich_minion(m):
    """Attach card_def key using the hearthstone library name lookup."""
    raw_name = card_name(m.get('card_id', '')) if m.get('card_id') else m.get('name', '')
    cdef = _NAME_TO_DEF.get(raw_name.lower())
    if cdef:
        sym_id = next((k for k, v in CARD_DEFS.items() if v['name'] == cdef['name']), m.get('card_id', ''))
        return dict(m, card_id=sym_id, _has_def=True)
    return dict(m, _has_def=False)

print(f"Card definitions loaded : {len(CARD_DEFS)}")
print(f"Card feature dimension  : {CARD_FEATURE_DIM}")
print("Symbolic components ready.")

### Symbolic Board Analysis

For each game we analyse the **final board** through the symbolic layer: multipliers, dominant tribe, aura dependency, and effect duration profile.

In [ ]:
def analyse_board(board, round_num, tier, gold=0):
    enriched = [enrich_minion(m) for m in board]
    return board_computer.compute(enriched, gold=gold, round_num=round_num, tavern_tier=tier)

rows_sym = []
for g in games:
    last_round = g['rounds'][-1]
    shopping   = last_round['shopping']
    combat     = last_round.get('combat') or {}
    board      = combat.get('player_board') or shopping.get('board_at_end', [])
    tier       = shopping.get('tavern_tier', 6)
    round_num  = last_round['round']
    gold       = shopping.get('gold_at_start', 10)
    hero       = card_name(g['hero']['card_id']) if g.get('hero') else '?'
    if not board:
        continue
    f = analyse_board(board, round_num, tier, gold)
    rows_sym.append({
        'hero': hero, 'round': round_num, 'tier': tier,
        'board_size': f.board_size, 'total_atk': f.total_atk, 'total_hp': f.total_hp,
        'brann': f.brann_active, 'titus': f.titus_active, 'drakkari': f.drakkari_active,
        'dominant_tribe': f.dominant_tribe or 'None',
        'is_synergistic': f.is_synergistic,
        'aura_dep': round(f.total_aura_dependency, 3),
        'perm_fx': f.permanent_effect_count, 'game_fx': f.this_game_count,
        'combat_fx': f.this_combat_count, 'dr_count': f.dr_count,
        'divine_shields': f.divine_shield_count, 'venomous': f.venomous_count,
        'reborn': f.reborn_count,
    })

df_sym = pd.DataFrame(rows_sym)
display(df_sym[['hero','round','tier','board_size','total_atk','total_hp',
                'dominant_tribe','is_synergistic','brann','titus','drakkari','aura_dep']])

### Effect Duration Profile

The symbolic layer tags every board effect as `PERMANENT`, `THIS_GAME`, or `THIS_COMBAT`. Permanent buffs are worth ~3× a combat-only buff of equivalent stats.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

heroes = df_sym['hero'].tolist()
perm   = df_sym['perm_fx'].tolist()
game   = df_sym['game_fx'].tolist()
cbt    = df_sym['combat_fx'].tolist()
x = range(len(heroes))

axes[0].bar(x, perm, label='Permanent',   color='#2ecc71')
axes[0].bar(x, game, bottom=perm, label='This Game', color='#3498db')
axes[0].bar(x, cbt,  bottom=[p+g for p,g in zip(perm,game)], label='This Combat', color='#e67e22')
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(heroes, rotation=25, ha='right', fontsize=9)
axes[0].set_ylabel('Effect count on final board')
axes[0].set_title('Effect Duration Profile — Final Boards')
axes[0].legend()

kw_labels = ['dr_count','divine_shields','venomous','reborn']
kw_colors = ['#9b59b6','#1abc9c','#e74c3c','#f39c12']
bottom = np.zeros(len(df_sym))
for label, color in zip(kw_labels, kw_colors):
    vals = df_sym[label].values.astype(float)
    axes[1].bar(x, vals, bottom=bottom, label=label.replace('_',' ').title(), color=color)
    bottom += vals
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(heroes, rotation=25, ha='right', fontsize=9)
axes[1].set_ylabel('Count')
axes[1].set_title('Board Keywords & Deathrattles — Final Boards')
axes[1].legend()
plt.tight_layout()
plt.show()

### Tribal Density Heatmap

A board is "synergistic" when ≥4 minions share a tribe. The heatmap shows per-tribe count for each game's final board.

In [ ]:
TRIBE_ORDER = ['BEAST','DEMON','DRAGON','ELEMENTAL','MECH',
               'MURLOC','NAGA','PIRATE','QUILBOAR','UNDEAD']

tribe_matrix = []
for g in games:
    last_round = g['rounds'][-1]
    shopping   = last_round['shopping']
    combat     = last_round.get('combat') or {}
    board      = combat.get('player_board') or shopping.get('board_at_end', [])
    tier       = shopping.get('tavern_tier', 6)
    round_num  = last_round['round']
    if not board:
        tribe_matrix.append([0]*len(TRIBE_ORDER))
        continue
    feats = analyse_board(board, round_num, tier)
    tribe_matrix.append([feats.tribe_counts.get(t, 0) for t in TRIBE_ORDER])

tribe_arr = np.array(tribe_matrix)
heroes_all = [card_name(g['hero']['card_id']) for g in games]

fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(tribe_arr.T, cmap='YlOrRd', aspect='auto', vmin=0, vmax=7)
ax.set_yticks(range(len(TRIBE_ORDER)))
ax.set_yticklabels(TRIBE_ORDER)
ax.set_xticks(range(len(heroes_all)))
ax.set_xticklabels(heroes_all, rotation=20, ha='right', fontsize=9)
ax.set_title('Tribal Composition of Final Boards')
plt.colorbar(im, ax=ax, label='Minion count')
for i in range(len(TRIBE_ORDER)):
    for j in range(len(heroes_all)):
        v = tribe_arr[j, i]
        if v > 0:
            ax.text(j, i, int(v), ha='center', va='center',
                    color='white' if v >= 5 else 'black', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Card Encoder — 44-Dim Feature Vector

Each minion is encoded as a **44-dimensional float32 vector**. Dims 0-11 are backward-compatible with the existing BC 11-dim encoding. Dims 12-43 add tribal identity, trigger types, effect durations, and board context.

In [ ]:
DIM_LABELS = (
    ['attack/20', 'health/20']
    + ['taunt','divine_shield','reborn','venomous','windfury','golden','magnetic']
    + ['tier/7', 'zone_pos/7', 'is_present']
    + [f'tribe_{t}' for t in TRIBE_LIST]
    + [f'trig_{t}' for t in TRIGGER_TYPES] + ['trig_other']
    + [f'dur_{t}' for t in DURATION_TYPES]
    + ['is_multiplier','is_aura','scales_with_board']
    + ['perm_atk/20','perm_hp/20','avenge_n/5','dr_summons']
    + ['board_size/7','synergy','aura_dep','round/25','tier_ctx/7']
)
assert len(DIM_LABELS) == 44

# Encode a sample board
DEMO_GAME_ENC  = 0
DEMO_ROUND_ENC = 10

g3 = games[DEMO_GAME_ENC]
r3 = g3['rounds'][DEMO_ROUND_ENC]
s3 = r3['shopping']
board3   = [enrich_minion(m) for m in s3.get('board_at_end', [])]
tier3    = s3.get('tavern_tier', 4)
round3   = r3['round']

feats3   = board_computer.compute(board3, round_num=round3, tavern_tier=tier3)
dom_cnt  = max(feats3.tribe_counts.values(), default=0)

enc_board = card_encoder.encode_board(
    board3,
    board_size=feats3.board_size,
    dominant_tribe_count=dom_cnt,
    total_aura_dependency=feats3.total_aura_dependency,
    round_num=round3, tavern_tier=tier3,
)
print(f"Encoded board shape : {enc_board.shape}  (7 slots × 44 features)")
print(f"Active slots        : {feats3.board_size}")
print()

for i, row in enumerate(enc_board[:feats3.board_size]):
    m = board3[i]
    name = card_name(m.get('card_id', '')) or '?'
    nz   = [(DIM_LABELS[j], round(float(row[j]),3)) for j in range(44) if abs(row[j]) > 1e-4]
    print(f'Slot {i}: {name}')
    for label, val in nz:
        print(f'  [{label:22s}] = {val:.3f}')
    print()

In [ ]:
# Feature heatmap across board slots
fig, ax = plt.subplots(figsize=(16, 5))
slot_labels = [
    (card_name(m.get('card_id','')) or '?')[:18]
    for m in board3[:7]
] + ['(pad)'] * (7 - len(board3[:7]))
im = ax.imshow(enc_board, cmap='RdYlGn', aspect='auto', vmin=-0.1, vmax=1.1)
ax.set_xticks(range(44))
ax.set_xticklabels(DIM_LABELS, rotation=90, fontsize=6)
ax.set_yticks(range(7))
ax.set_yticklabels(slot_labels, fontsize=9)
hero3 = card_name(g3['hero']['card_id'])
ax.set_title(f'44-Dim Card Encoding — {hero3}  Round {round3}  Tier {tier3}')
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

## BC v2 — Two-headed policy (action type + card pointer)

Instead of predicting an absolute shop/board slot index, the model predicts:
1. **Action type** (8 classes: buy/sell/place/reroll/freeze/level_up/hero_power/end_turn)
2. **Card pointer** (24 slots: shop[0-6] + board[0-6] + hand[0-9])

This separates *what to do* from *which card*, so the type head generalises across games regardless of shop ordering.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from collections import Counter

# ── V2 action space ───────────────────────────────────────────────────────────
N_ACTION_TYPES    = 8
ACTION_TYPE_NAMES = ["buy", "sell", "place", "reroll", "freeze", "level_up", "hero_power", "end_turn"]

SHOP_ZONE_SIZE  = 7
BOARD_ZONE_SIZE = 7
HAND_ZONE_SIZE  = 10
POINTER_DIM     = SHOP_ZONE_SIZE + BOARD_ZONE_SIZE + HAND_ZONE_SIZE  # 24

_PTR_SHOP_OFF  = 0
_PTR_BOARD_OFF = SHOP_ZONE_SIZE
_PTR_HAND_OFF  = SHOP_ZONE_SIZE + BOARD_ZONE_SIZE

print(f"Action types  : {N_ACTION_TYPES}  {ACTION_TYPE_NAMES}")
print(f"Pointer slots : {POINTER_DIM}  (shop:{SHOP_ZONE_SIZE} + board:{BOARD_ZONE_SIZE} + hand:{HAND_ZONE_SIZE})")


In [ ]:
# ── State vector layout ───────────────────────────────────────────────────────
# Context (7): round/20, tier/6, health/40, armor/40, gold/10, hp_cost/5, hp_avail
# Shop    (7 × 11 = 77)
# Board   (7 × 11 = 77)
# Hand    (10 × 11 = 110)
# Total   : 7 + 77 + 77 + 110 = 271
CARD_FEATS    = 11
MAX_SHOP      = 7
MAX_BOARD     = 7
MAX_HAND      = 10
CONTEXT_FEATS = 7
STATE_DIM     = CONTEXT_FEATS + (MAX_SHOP + MAX_BOARD + MAX_HAND) * CARD_FEATS  # 271

_SHOP_OFF  = CONTEXT_FEATS
_BOARD_OFF = _SHOP_OFF  + MAX_SHOP  * CARD_FEATS
_HAND_OFF  = _BOARD_OFF + MAX_BOARD * CARD_FEATS
_IS_PRES   = CARD_FEATS - 1   # index of is_present within each slot (= 10)

assert STATE_DIM == 271, STATE_DIM


def encode_card(m: dict) -> list:
    """Encode one minion/spell as an 11-dim float vector."""
    return [
        m.get("attack", 0)             / 10.0,
        m.get("health", 0)             / 10.0,
        float(m.get("divine_shield",  False)),
        float(m.get("taunt",          False)),
        float(m.get("reborn",         False)),
        float(m.get("poisonous",      False)),
        float(m.get("windfury",       False)),
        float(m.get("golden",         False)),
        m.get("tier", 1)               /  6.0,
        m.get("zone_pos", 0)           /  7.0,
        1.0,   # is_present
    ]

_EMPTY_CARD = [0.0] * CARD_FEATS

def encode_slot_list(minions: list, max_slots: int) -> list:
    feats = []
    for i in range(max_slots):
        feats.extend(encode_card(minions[i]) if i < len(minions) else _EMPTY_CARD)
    return feats

def encode_state(round_num: int, tier: int, health: int, armor: int,
                 shop: list, board: list, hand: list, gold: int = 0,
                 hero_power_cost: int = 0, hero_power_available: bool = True) -> list:
    context = [
        round_num         / 20.0,
        tier              /  6.0,
        health            / 40.0,
        armor             / 40.0,
        gold              / 10.0,
        hero_power_cost   /  5.0,
        float(hero_power_available),
    ]
    return (context
            + encode_slot_list(shop,  MAX_SHOP)
            + encode_slot_list(board, MAX_BOARD)
            + encode_slot_list(hand,  MAX_HAND))

def find_by_card_id(lst: list, card_id: str) -> int:
    """Return index of first entry with matching card_id, or -1."""
    for i, m in enumerate(lst):
        if m.get("card_id") == card_id:
            return i
    return -1

print(f"STATE_DIM = {STATE_DIM}")
print(f"encode_state output length: {len(encode_state(1, 1, 30, 0, [], [], []))}  (expected {STATE_DIM})")


In [ ]:
def extract_transitions_v2(games: list):
    """
    V2 extraction: returns two label arrays per transition.
      ya  — action TYPE label  (0-7)
      yc  — card POINTER label (0-23), or -1 for non-card actions / unknown slot
    Card pointer mapping:
      buy   -> shop slot i    -> _PTR_SHOP_OFF  + i  (0-6)
      sell  -> board slot i   -> _PTR_BOARD_OFF + i  (7-13)
      place -> hand slot i    -> _PTR_HAND_OFF  + i  (14-23)
      all others              -> -1
    """
    X, ya, yc, game_ids = [], [], [], []
    skipped_buys = 0

    type_map = {"buy": 0, "sell": 1, "place": 2, "reroll": 3,
                "freeze": 4, "level_up": 5, "hero_power": 6, "end_turn": 7}

    for gidx, game in enumerate(games):
        prev_hand: list = []
        for r in game.get("rounds", []):
            shopping = r.get("shopping")
            if not shopping:
                continue

            round_num = r["round"]
            tier      = shopping.get("tavern_tier", 1)
            health    = shopping.get("hero_health", 30)
            armor     = shopping.get("hero_armor",   0)
            hp_cost   = shopping.get("hero_power_cost", 0)
            actions   = shopping.get("actions", [])

            reroll_shops: dict = {}
            for i, a in enumerate(actions):
                if a.get("action") == "reroll":
                    new_cids = [b["card_id"] for b in actions[i + 1:]
                                if b.get("action") == "buy" and b.get("card_id")]
                    reroll_shops[i] = [{"card_id": c} for c in new_cids]

            shop  = list(shopping.get("shop_at_start",  []))
            board = list(shopping.get("board_at_start", []))
            hand  = list(prev_hand)
            gold  = shopping.get("gold_at_start", 0)
            hp_used = False

            for ai, action in enumerate(actions):
                atype_s = action.get("action", "")
                cid     = action.get("card_id", "")
                gold    = action.get("gold_remaining", gold)

                type_label = type_map.get(atype_s)
                if type_label is None:
                    continue

                state     = encode_state(round_num, tier, health, armor, shop, board, hand,
                                         gold, hero_power_cost=hp_cost,
                                         hero_power_available=not hp_used)
                ptr_label = -1

                if atype_s == "buy":
                    idx = find_by_card_id(shop, cid)
                    if idx == -1:
                        skipped_buys += 1
                        if cid:
                            hand.append({"card_id": cid})
                        gold = max(0, gold - 3)
                        continue
                    ptr_label = _PTR_SHOP_OFF + min(idx, 6)
                    hand.append(shop.pop(idx))
                    gold -= 3

                elif atype_s == "sell":
                    idx = find_by_card_id(board, cid)
                    if idx != -1:
                        ptr_label = _PTR_BOARD_OFF + min(idx, 6)
                        board.pop(idx)
                    gold += 1

                elif atype_s == "place":
                    idx = find_by_card_id(hand, cid)
                    if idx != -1:
                        ptr_label = _PTR_HAND_OFF + min(idx, 9)
                        board.append(hand.pop(idx))

                elif atype_s == "reroll":
                    shop = list(reroll_shops.get(ai, []))
                    gold -= 1

                elif atype_s == "level_up":
                    tier = min(tier + 1, 6)

                elif atype_s == "hero_power":
                    hp_used = True
                    gold -= action.get("hero_power_cost", hp_cost)

                gold = max(0, gold)
                X.append(state); ya.append(type_label); yc.append(ptr_label)
                game_ids.append(gidx)

            prev_hand = list(shopping.get("hand_at_end", hand))

            X.append(encode_state(round_num, tier, health, armor, shop, board, hand,
                                  gold, hero_power_cost=hp_cost,
                                  hero_power_available=not hp_used))
            ya.append(type_map["end_turn"]); yc.append(-1); game_ids.append(gidx)

    return (np.array(X,  dtype=np.float32),
            np.array(ya, dtype=np.int64),
            np.array(yc, dtype=np.int64),
            np.array(game_ids, dtype=np.int64),
            skipped_buys)


X2_all, ya_all, yc_all, gids2, n_skip2 = extract_transitions_v2(games)

print(f"Total transitions   : {len(X2_all)}")
print(f"Buys skipped        : {n_skip2}")
print(f"Pointer known (!=-1): {(yc_all != -1).sum()}")
print()
print("Action type distribution:")
tc = Counter(int(a) for a in ya_all)
for i, name in enumerate(ACTION_TYPE_NAMES):
    cnt = tc.get(i, 0)
    print(f"  {name:12s}: {cnt:4d}  ({cnt/len(ya_all):.1%})")


In [ ]:
_LEVEL_COST_V2 = {1: 5, 2: 7, 3: 8, 4: 9, 5: 10}

def valid_action_type_mask(state) -> np.ndarray:
    s = np.asarray(state, dtype=np.float32)
    gold     = int(round(float(s[4]) * 10))
    tier     = int(round(float(s[1]) * 6))
    hp_cost  = int(round(float(s[5]) * 5))
    hp_avail = float(s[6]) > 0.5
    n_shop  = int(sum(s[_SHOP_OFF  + i*CARD_FEATS + _IS_PRES] > 0.5 for i in range(MAX_SHOP)))
    n_board = int(sum(s[_BOARD_OFF + i*CARD_FEATS + _IS_PRES] > 0.5 for i in range(MAX_BOARD)))
    n_hand  = int(sum(s[_HAND_OFF  + i*CARD_FEATS + _IS_PRES] > 0.5 for i in range(MAX_HAND)))
    mask = np.zeros(N_ACTION_TYPES, dtype=bool)
    if n_shop > 0 and gold >= 3 and n_hand < MAX_HAND: mask[0] = True
    if n_board > 0:                                     mask[1] = True
    if n_hand > 0 and n_board < MAX_BOARD:              mask[2] = True
    if gold >= 1:                                       mask[3] = True
    mask[4] = True
    if tier < 6 and gold >= _LEVEL_COST_V2.get(tier, 99): mask[5] = True
    if hp_avail and gold >= hp_cost:                    mask[6] = True
    mask[7] = True
    return mask

def compute_type_masks(X, y=None):
    masks = np.stack([valid_action_type_mask(x).astype(np.float32) for x in X])
    if y is not None:
        masks[np.arange(len(y)), y] = 1.0
    return masks

_s = encode_state(1, 1, 30, 0,
                  [{"card_id":"X","attack":1,"health":1,"tier":1}]*3,
                  [{"card_id":"Y","attack":2,"health":2,"tier":1}]*2,
                  [], gold=5, hero_power_cost=2)
_m = valid_action_type_mask(_s)
print("Valid types (3-shop, 2-board, empty hand, 5g, hp_cost=2):")
print(" ", [ACTION_TYPE_NAMES[i] for i, v in enumerate(_m) if v])


In [ ]:
class BGPolicyV2(nn.Module):
    """
    Two-headed BC policy.
      type_head    : state (271) -> 8 action-type logits
      pointer_head : state (271) -> 24 card-pointer logits
                     layout: shop[0-6] | board[0-6] | hand[0-9]
    """
    def __init__(self, state_dim: int = STATE_DIM, hidden: int = 128):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.LayerNorm(hidden), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden, hidden),    nn.LayerNorm(hidden), nn.ReLU(), nn.Dropout(0.2),
        )
        self.type_head    = nn.Linear(hidden, N_ACTION_TYPES)
        self.pointer_head = nn.Linear(hidden, POINTER_DIM)

    def forward(self, x: torch.Tensor):
        h = self.shared(x)
        return self.type_head(h), self.pointer_head(h)


model_v2 = BGPolicyV2()
print(f"BGPolicyV2 parameters: {sum(p.numel() for p in model_v2.parameters()):,}")


In [ ]:
n_games2  = len(games)
n_val2    = max(1, int(0.2 * n_games2))
val_ids2  = set(range(n_games2 - n_val2, n_games2))
tmask2    = np.array([g not in val_ids2 for g in gids2])

X2_tr, ya_tr, yc_tr = X2_all[tmask2],  ya_all[tmask2],  yc_all[tmask2]
X2_va, ya_va, yc_va = X2_all[~tmask2], ya_all[~tmask2], yc_all[~tmask2]

print(f"Train: {len(X2_tr)} | Val: {len(X2_va)}")

mt_tr       = compute_type_masks(X2_tr, ya_tr)
mt_va       = compute_type_masks(X2_va)
mt_va_sched = compute_type_masks(X2_va, ya_va)

EPOCHS_V2  = 300
BATCH_V2   = 32
LR_V2      = 1e-3
NEG_INF_V2 = -1e9
PTR_WEIGHT = 0.5

dl_v2 = DataLoader(
    TensorDataset(torch.from_numpy(X2_tr), torch.from_numpy(ya_tr),
                  torch.from_numpy(yc_tr), torch.from_numpy(mt_tr)),
    batch_size=BATCH_V2, shuffle=True)

opt_v2   = optim.AdamW(model_v2.parameters(), lr=LR_V2, weight_decay=1e-3)
crit_v2  = nn.CrossEntropyLoss()
sched_v2 = optim.lr_scheduler.ReduceLROnPlateau(opt_v2, patience=40, factor=0.5)

best_tacc = 0.0
for epoch in range(1, EPOCHS_V2 + 1):
    model_v2.train()
    ep_loss = 0.0
    for xb, yab, ycb, mb in dl_v2:
        opt_v2.zero_grad()
        t_log, p_log = model_v2(xb)
        loss = crit_v2(t_log + (1 - mb) * NEG_INF_V2, yab)
        ep_loss += loss.item() * len(xb)
        ptr_valid = (ycb != -1).nonzero(as_tuple=True)[0]
        if len(ptr_valid) > 0:
            loss = loss + PTR_WEIGHT * crit_v2(p_log[ptr_valid], ycb[ptr_valid])
        loss.backward(); opt_v2.step()

    model_v2.eval()
    with torch.no_grad():
        vt, _ = model_v2(torch.from_numpy(X2_va))
        val_sched = crit_v2(vt + (1 - torch.from_numpy(mt_va_sched)) * NEG_INF_V2,
                            torch.from_numpy(ya_va)).item()
        val_tacc  = (( vt + (1 - torch.from_numpy(mt_va)) * NEG_INF_V2
                     ).argmax(1).numpy() == ya_va).mean()
    best_tacc = max(best_tacc, val_tacc)
    sched_v2.step(val_sched)

    if epoch % 50 == 0:
        print(f"Epoch {epoch:3d}/{EPOCHS_V2}  "
              f"type_loss={ep_loss/len(X2_tr):.4f}  val_type_acc={val_tacc:.1%}")

print(f"\nBest val type accuracy : {best_tacc:.1%}")
majority2 = Counter(int(a) for a in ya_va).most_common(1)[0][1] / len(ya_va)
print(f"Majority-type baseline : {majority2:.1%}")


In [ ]:
model_v2.eval()
with torch.no_grad():
    t_all, p_all = model_v2(torch.from_numpy(X2_all))
    mt_all = compute_type_masks(X2_all)
    t_preds = (t_all + (1 - torch.from_numpy(mt_all)) * NEG_INF_V2).argmax(1).numpy()
    p_preds = p_all.argmax(1).numpy()

print("Per-type accuracy (full dataset):")
for i, name in enumerate(ACTION_TYPE_NAMES):
    mask  = ya_all == i
    total = mask.sum()
    if not total: continue
    correct = (t_preds[mask] == i).sum()
    print(f"  {name:12s}: {correct:4d}/{total:4d} = {correct/total:.1%}")

ptr_mask = yc_all != -1
if ptr_mask.sum() > 0:
    correct_ptr = (p_preds[ptr_mask] == yc_all[ptr_mask]).sum()
    print(f"\nCard pointer accuracy (slot-known): {correct_ptr}/{ptr_mask.sum()} = {correct_ptr/ptr_mask.sum():.1%}")
    for zone_name, off, size in [("buy/shop",  _PTR_SHOP_OFF,  SHOP_ZONE_SIZE),
                                  ("sell/board",_PTR_BOARD_OFF, BOARD_ZONE_SIZE),
                                  ("place/hand",_PTR_HAND_OFF,  HAND_ZONE_SIZE)]:
        zm = ptr_mask & (yc_all >= off) & (yc_all < off + size)
        if zm.sum() == 0: continue
        zc = (p_preds[zm] == yc_all[zm]).sum()
        print(f"  {zone_name:12s}: {zc}/{zm.sum()} = {zc/zm.sum():.1%}")


## BC v2 — Save checkpoint for PPO warm-start

In [ ]:
from pathlib import Path

BC_V2_PATH = Path("bc_v2.pt")

torch.save({
    "state_dict"   : model_v2.state_dict(),
    "state_dim"    : STATE_DIM,
    "n_action_types": N_ACTION_TYPES,
    "pointer_dim"  : POINTER_DIM,
    "hidden"       : 128,
    "action_type_names": ACTION_TYPE_NAMES,
    # Explicit group mapping: BC type index -> PPO action index range (inclusive)
    # Used by BGPolicyNetwork.load_bc_v2_weights() in agent/policy.py
    "ppo_action_groups": {
        0: list(range(0,  7)),   # buy   -> buy_0..buy_6
        1: list(range(7,  14)),  # sell  -> sell_0..sell_6
        2: list(range(14, 84)),  # place -> play_h*_p*
        3: [86],                 # reroll -> refresh (PPO index 86)
        4: [85],                 # freeze -> freeze  (PPO index 85)
        5: [84],                 # level_up           (PPO index 84)
        6: [87],                 # hero_power          (PPO index 87)
        7: [88],                 # end_turn            (PPO index 88)
    },
}, BC_V2_PATH)

print(f"Saved BC v2 checkpoint → {BC_V2_PATH}")
print(f"  shared trunk : {sum(p.numel() for p in model_v2.shared.parameters()):,} params")
print(f"  type_head    : {sum(p.numel() for p in model_v2.type_head.parameters()):,} params")
print(f"  pointer_head : {sum(p.numel() for p in model_v2.pointer_head.parameters()):,} params")


---
## PPO Policy Network

`BGPolicyNetwork` uses a Transformer encoder over **32 tokens**
(`[CLS]` + 7 board + 7 shop + 10 hand + 7 opponent board), fused with a
**38-dim scalar context**, feeding **two separate heads** matching `BGPolicyV2`:

- **`type_head`** → 8 action-type logits (buy / sell / place / reroll / freeze / level_up / hero_power / end_turn)
- **`pointer_head`** → 24 card-pointer logits (shop[0-6] | board[0-6] | hand[0-9])
- **`value_head`** → scalar state value

This two-step structure means BC weights transfer **directly** via
`policy.load_bc_v2_weights("bc_v2.pt")` — no row-mapping required.


In [ ]:
import torch
from agent.policy import BGPolicyNetwork, build_type_mask, build_pointer_mask
from agent.policy import N_ACTION_TYPES, ACTION_TYPE_NAMES, POINTER_DIM, SCALAR_DIM

policy = BGPolicyNetwork()   # card_dim=44, d_model=128, nhead=4, num_layers=3
n_params = sum(p.numel() for p in policy.parameters())
print(f"PPO Policy parameters : {n_params:,}")
print(f"Action types          : {N_ACTION_TYPES}  {ACTION_TYPE_NAMES}")
print(f"Pointer slots         : {POINTER_DIM}")
print(f"Scalar context dim    : {SCALAR_DIM}")
print()
# Warm-start from BGPolicyV2 BC checkpoint (uncomment after running BC v2 training)
policy.load_bc_v2_weights("bc_v2.pt")
print(policy)


In [ ]:
def enc_zone(zone, tavern_tier, round_num):
    """Encode a zone (board/shop/hand) -> (7×44 array, BoardFeatures)."""
    feats    = board_computer.compute(zone, round_num=round_num, tavern_tier=tavern_tier)
    dom_cnt  = max(feats.tribe_counts.values(), default=0)
    encoded  = card_encoder.encode_board(
        zone,
        board_size=feats.board_size,
        dominant_tribe_count=dom_cnt,
        total_aura_dependency=feats.total_aura_dependency,
        round_num=round_num,
        tavern_tier=tavern_tier,
    )
    return encoded, feats


In [ ]:
# Forward pass demo with a real shopping state
g4 = games[0]
r4 = g4['rounds'][5]
s4 = r4['shopping']

board4 = [enrich_minion(m) for m in s4.get('board_at_end', [])]
shop4  = [enrich_minion(m) for m in s4.get('shop_at_start', [])]
hand4  = []
tier4  = s4.get('tavern_tier', 2)
round4 = r4['round']
gold4  = s4.get('gold_at_start', 5)

board_enc4, bfeats4 = enc_zone(board4, tier4, round4)
shop_enc4,  _       = enc_zone(shop4,  tier4, round4)
hand_enc4,  _       = enc_zone(hand4,  tier4, round4)

board_t = torch.tensor(board_enc4, dtype=torch.float32).unsqueeze(0)  # [1, 7, 44]
shop_t  = torch.tensor(shop_enc4,  dtype=torch.float32).unsqueeze(0)  # [1, 7, 44]

# encode_board always returns [7, 44]; pad hand to exactly 10 slots
hand_arr = np.zeros((10, 44), dtype=np.float32)
hand_arr[:min(len(hand_enc4), 10)] = hand_enc4[:10]
hand_t = torch.tensor(hand_arr, dtype=torch.float32).unsqueeze(0)     # [1, 10, 44]

# 38-dim scalar: own (24) + opponent/lobby (14, zeros for demo)
own_scalar = bfeats4.to_scalar_vector()
scalar_t   = torch.tensor(
    np.concatenate([own_scalar, np.zeros(14)]), dtype=torch.float32
).unsqueeze(0)  # [1, 38]

# Build two separate masks
ps4 = {'gold': gold4, 'shop': shop4, 'board': board4, 'hand': hand4,
       'tavern_tier': tier4, 'level_cost': 5}
t_mask = build_type_mask(ps4)
p_mask = build_pointer_mask(ps4, -1)   # full occupancy, all zones

print(f'Board: {len(board4)} minions | Shop: {len(shop4)} cards | Gold: {gold4} | Tier: {tier4}')
print(f'Valid types ({t_mask.sum().item()}): {[ACTION_TYPE_NAMES[i] for i,v in enumerate(t_mask) if v]}')
print()

# Two-step forward pass
policy.eval()
with torch.no_grad():
    type_logits, ptr_logits, value = policy(
        board_t, shop_t, hand_t, scalar_t,
        type_mask=t_mask.unsqueeze(0), pointer_mask=p_mask.unsqueeze(0)
    )
    type_probs = torch.softmax(type_logits.squeeze(0), dim=-1)
    ptr_probs  = torch.softmax(ptr_logits.squeeze(0),  dim=-1)

print(f'State value estimate : {value.item():.4f}')
print('\nAction type probabilities (untrained — random init):')
for i, (name, prob) in enumerate(zip(ACTION_TYPE_NAMES, type_probs)):
    if t_mask[i]:
        bar = chr(9608) * int(prob.item() * 40)
        print(f'  {name:<12s} {prob.item():.3f}  {bar}')

# Two-step sample
type_idx, ptr_idx, log_prob, val = policy.get_action(
    board_t, shop_t, hand_t, scalar_t,
    type_mask=t_mask.unsqueeze(0), pointer_mask=p_mask.unsqueeze(0)
)
ptr_names = ([f'shop_{i}' for i in range(7)] +
             [f'board_{i}' for i in range(7)] +
             [f'hand_{i}' for i in range(10)])
ptr_str = ptr_names[ptr_idx] if ptr_idx >= 0 else 'n/a'
print(f'\nSampled: type={ACTION_TYPE_NAMES[type_idx]}  ptr={ptr_str}  log_prob={log_prob.item():.3f}')


In [ ]:
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(15, 7))
ax.set_xlim(0, 15); ax.set_ylim(0, 7); ax.axis('off')
ax.set_facecolor('#f8f9fa'); fig.patch.set_facecolor('#f8f9fa')

def box(x, y, w, h, text, color, fs=9):
    r = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                       lw=1, edgecolor='#555', facecolor=color, alpha=0.9)
    ax.add_patch(r)
    ax.text(x+w/2, y+h/2, text, ha='center', va='center', fontsize=fs, fontweight='bold')

def arrow(x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#444', lw=1.2))

# Input token zones
box(0.2, 5.7, 3.0, 0.8, 'Board Tokens\n[7 x 44]',   '#AED6F1')
box(0.2, 4.6, 3.0, 0.8, 'Shop Tokens\n[7 x 44]',    '#A9DFBF')
box(0.2, 3.5, 3.0, 0.8, 'Hand Tokens\n[10 x 44]',   '#FAD7A0')
box(0.2, 2.4, 3.0, 0.8, 'Opp Tokens\n[7 x 44]',     '#F1948A')
box(0.2, 0.5, 3.0, 0.8, 'Scalar Context\n[38]',     '#D7BDE2')

# Linear proj + zone embed
box(4.0, 3.2, 1.8, 3.0, 'Linear\nProj +\nZone\nEmbed', '#D5DBDB', 8)
arrow(3.2, 6.1, 4.0, 5.2)
arrow(3.2, 5.0, 4.0, 4.9)
arrow(3.2, 3.9, 4.0, 4.3)
arrow(3.2, 2.8, 4.0, 3.7)

# Transformer encoder
box(6.3, 1.5, 2.4, 4.7, 'Transformer\nEncoder\n(3 layers)\n[32 x 128]\nCLS+7b+7s\n+10h+7o', '#FDEBD0', 8)
arrow(5.8, 4.7, 6.3, 3.8)
ax.text(6.55, 6.5, 'CLS ->', fontsize=8, color='#555')
box(6.3, 6.2, 1.8, 0.6, 'CLS [128]', '#F9E79F', 8)

# Scalar projection
box(4.0, 0.5, 1.8, 0.8, 'Linear\nProject', '#D5DBDB', 8)
arrow(3.2, 0.9, 4.0, 0.9)
arrow(4.9, 0.9, 9.4, 0.9)

# Concat & fuse
box(9.4, 2.5, 1.8, 4.2, 'Concat\n[CLS||scl]\n256-dim', '#E8DAEF', 8)
arrow(8.7, 3.8, 9.4, 4.6)
arrow(7.2, 6.2, 9.4, 5.5)

# Three output heads — matching BGPolicyV2
box(11.6, 5.2, 2.6, 0.9, 'type_head\n[8 action types]',   '#ABEBC6')
box(11.6, 3.8, 2.6, 0.9, 'pointer_head\n[24 card slots]', '#A9DFBF')
box(11.6, 2.4, 2.6, 0.9, 'value_head\n[1]',               '#F1948A')
arrow(11.2, 5.0, 11.6, 5.65)
arrow(11.2, 4.2, 11.6, 4.25)
arrow(11.2, 3.4, 11.6, 2.85)

# BC warm-start annotation
ax.annotate('BC v2 weights\ntransfer directly\n(same shapes)',
            xy=(11.9, 5.15), fontsize=7.5, color='#555',
            ha='center',
            bbox=dict(boxstyle='round,pad=0.2', fc='#fffde7', ec='#bbb'))

ax.set_title('BGPolicyNetwork — Two-Headed Transformer (matches BGPolicyV2 BC output structure)',
             fontsize=12, fontweight='bold', pad=10)
plt.tight_layout()
plt.show()


### 38-Dim Scalar Context Vector

Beyond the card tokens, the policy receives a **38-dim scalar summary** split across three groups:

| Slice | Dims | Contents |
|---|---|---|
| Own board | [0:24] | win probability, multiplier flags (Brann/Titus/Drakkari), tribal densities x10, aura dependency, deathrattle count, synergy flag, board metrics |
| Next opponent | [24:32] | tier, health, armor, board size, dominant tribe count, is\_synergistic, rounds since last seen, health delta |
| Lobby-wide | [32:38] | num alive, mean opponent tier/health, num synergistic boards, health rank, tier rank |

Opponent and lobby dims are zeroed in notebook demos; the game loop populates them from `PlayerState` during training.


In [ ]:
SCALAR_LABELS = [
    # own board (24)
    'win_prob', 'dmg_dealt/40', 'dmg_taken/40',
    'brann', 'titus', 'drakkari',
    'tribe_BEAST', 'tribe_DEMON', 'tribe_DRAGON', 'tribe_ELEMENTAL',
    'tribe_MECH', 'tribe_MURLOC', 'tribe_NAGA', 'tribe_PIRATE',
    'tribe_QUILBOAR', 'tribe_UNDEAD',
    'aura_dep', 'dr_cnt/7', 'synergistic',
    'board_sz/7', 'div_shields/7', 'venomous/7',
    'slot_22', 'slot_23',
    # next opponent (8)
    'opp_tier/7', 'opp_health/40', 'opp_armor/10', 'opp_board_sz/7',
    'opp_dom_tribe/7', 'opp_synergistic', 'opp_rounds_since/10', 'opp_health_delta/40',
    # lobby-wide (6)
    'num_alive/8', 'mean_opp_tier/7', 'mean_opp_health/40',
    'num_synergistic/7', 'health_rank/8', 'tier_rank/8',
]
assert len(SCALAR_LABELS) == 38

all_scalars, all_meta = [], []
for g in games:
    hero = card_name(g['hero']['card_id'])
    for r in g['rounds'][::3]:
        s = r['shopping']
        board = [enrich_minion(m) for m in s.get('board_at_end', [])]
        if not board:
            continue
        f = board_computer.compute(board, round_num=r['round'], tavern_tier=s.get('tavern_tier', 1))
        own_scalar  = f.to_scalar_vector()                       # 24-dim
        full_scalar = np.concatenate([own_scalar, np.zeros(14)]) # pad opp + lobby for demo
        all_scalars.append(full_scalar)
        all_meta.append(f'{hero[:10]} r{r["round"]}')

scalar_matrix = np.stack(all_scalars)
fig, ax = plt.subplots(figsize=(16, max(4, len(all_scalars) * 0.4 + 1)))
im = ax.imshow(scalar_matrix, cmap='coolwarm', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(38))
ax.set_xticklabels(SCALAR_LABELS, rotation=70, ha='right', fontsize=7)
ax.set_yticks(range(len(all_meta)))
ax.set_yticklabels(all_meta, fontsize=8)
ax.axvline(23.5, color='#333', lw=1.5, ls='--')   # own | opp boundary
ax.axvline(31.5, color='#333', lw=1.5, ls='--')   # opp | lobby boundary
ax.set_title('38-Dim Scalar Context Vector - Sampled Rounds\n(opponent + lobby dims zeroed in this demo)')
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()


---
## PPO Training (from BC warm-start)

Train the PPO agent via self-play, starting from the pre-trained BC v2 weights.

The loop runs **N games** with shared-policy agents, collecting transitions into
a PPO buffer that is updated every `update_interval` games.
Uses the `--no-firestone` heuristic combat estimator for speed.

In [ ]:
import time, json, random, logging
from pathlib import Path
from itertools import groupby

import numpy as np
import torch
import matplotlib.pyplot as plt

from agent.card_encoder import CardEncoder
from agent.policy import (BGPolicyNetwork, build_type_mask, build_pointer_mask,
                           N_ACTION_TYPES)
from agent.ppo import PPOConfig, PPOTrainer
from env.game_loop import BattlegroundsGame
from env.matchmaker import Matchmaker
from env.tavern_pool import TavernPool
from symbolic.board_computer import SymbolicBoardComputer
from symbolic.firestone_client import FirestoneClient

# Silence noisy sub-loggers during training
for _ln in ('env.game_loop', 'env.tavern_pool', 'agent.ppo',
            'symbolic.firestone_client', 'symbolic.effect_handler'):
    logging.getLogger(_ln).setLevel(logging.WARNING)

DEVICE = 'cpu'
N_PLAYERS = 8
BC_V2_PATH = Path('bc_v2.pt')
PPO_PATH   = Path('bg_agent_ppo.pt')

with open('bg_card_definitions.json', encoding='utf-8') as _f:
    _raw = json.load(_f)
if isinstance(_raw, dict) and 'cards' in _raw:
    _raw = _raw['cards']
if isinstance(_raw, list):
    _raw = {d.get('card_id', str(i)): d for i, d in enumerate(_raw)}
card_defs_train = _raw
print(f'Loaded {len(card_defs_train)} card defs')

In [ ]:
policy_train = BGPolicyNetwork(
    card_dim=44, d_model=128, nhead=4, num_layers=3, scalar_dim=38, dropout=0.1
).to(DEVICE)

ppo_config = PPOConfig(
    lr=3e-4, gamma=0.99, gae_lambda=0.95,
    clip_eps=0.2, value_coef=0.5, entropy_coef=0.02,
    n_epochs=4, batch_size=64, device=DEVICE,
)
ppo_trainer = PPOTrainer(policy_train, ppo_config)

if BC_V2_PATH.exists():
    policy_train.load_bc_v2_weights(str(BC_V2_PATH))
    print(f'Loaded BC v2 weights from {BC_V2_PATH}')
else:
    print(f'WARNING: {BC_V2_PATH} not found — training from scratch')

if PPO_PATH.exists():
    ppo_trainer.load_checkpoint(str(PPO_PATH))
    print(f'Resumed PPO: steps={ppo_trainer.total_steps}, updates={ppo_trainer.update_count}')
else:
    print('No PPO checkpoint — starting from step 0')

In [ ]:
ACTION_NAMES = ['BUY', 'SELL', 'PLACE', 'REROLL', 'FREEZE', 'LEVEL', 'HERO_PWR', 'END_TURN']

class _Agent:
    def __init__(self, policy, trainer, pid, device='cpu'):
        self.policy, self.trainer, self.pid, self.device = policy, trainer, pid, device
        self._last_obs = None
        self._cached_type_mask = None
        self._cached_ptr_mask  = None

    def get_action(self, obs):
        self._last_obs = obs
        ps  = obs['player_state']
        dev = torch.device(self.device)
        # Cache masks NOW before step_shopping mutates the live ps object
        self._cached_type_mask = build_type_mask(ps).numpy()
        t_mask = torch.from_numpy(self._cached_type_mask).unsqueeze(0).to(dev)
        p_mask = build_pointer_mask(ps, -1).unsqueeze(0).to(dev)
        bt = torch.tensor(obs['board_tokens'][None],   dtype=torch.float32, device=dev)
        st = torch.tensor(obs['shop_tokens'][None],    dtype=torch.float32, device=dev)
        ht = torch.tensor(obs['hand_tokens'][None],    dtype=torch.float32, device=dev)
        sc = torch.tensor(obs['scalar_context'][None], dtype=torch.float32, device=dev)
        op = obs.get('opp_tokens')
        ot = torch.tensor(op[None], dtype=torch.float32, device=dev) if op is not None else None
        type_idx, ptr_idx, _, _ = self.policy.get_action(
            bt, st, ht, sc, type_mask=t_mask, pointer_mask=p_mask,
            deterministic=False, opp_tokens=ot,
        )
        self._cached_ptr_mask = build_pointer_mask(ps, type_idx).numpy()
        return type_idx, ptr_idx

    def record_transition(self, obs, ta, pa, reward, done):
        if obs is None:
            return
        self.trainer.collect_transition(
            board_tokens=obs['board_tokens'], shop_tokens=obs['shop_tokens'],
            hand_tokens=obs['hand_tokens'],  scalar_context=obs['scalar_context'],
            type_action=ta, ptr_action=pa,
            type_mask=self._cached_type_mask,
            pointer_mask=self._cached_ptr_mask,
            reward=reward, done=done, opp_tokens=obs.get('opp_tokens'),
        )

print('Agent wrapper ready.')

In [ ]:
N_GAMES         = 100   # games to run per cell execution; re-run to keep training
UPDATE_INTERVAL = 10

# Seed offset from total steps so re-runs use fresh game seeds automatically
_seed_offset = ppo_trainer.total_steps
random.seed(_seed_offset); np.random.seed(_seed_offset); torch.manual_seed(_seed_offset)

# Accumulate history across re-runs (lists persist in notebook state)
if 'game_rewards' not in dir() or not isinstance(game_rewards, list):
    game_rewards, ppo_losses, ppo_values = [], [], []

for game_idx in range(1, N_GAMES + 1):
    _seed = _seed_offset + game_idx
    tavern_pool = TavernPool(card_defs_train, seed=_seed)
    matchmaker  = Matchmaker(n_players=N_PLAYERS, seed=_seed)
    board_comp  = SymbolicBoardComputer(card_defs_train)
    firestone   = FirestoneClient(firestone_path=None, mock_mode=True)
    agents = [_Agent(policy_train, ppo_trainer, pid, DEVICE) for pid in range(N_PLAYERS)]

    game = BattlegroundsGame(
        card_defs=card_defs_train, agents=agents,
        board_computer=board_comp, firestone_client=firestone,
        matchmaker=matchmaker, tavern_pool=tavern_pool,
        n_players=N_PLAYERS, seed=_seed,
    )
    result = game.run_game()
    mean_r = float(np.mean(list(result.final_rewards.values())))
    game_rewards.append(mean_r)

    if game_idx % UPDATE_INTERVAL == 0 and len(ppo_trainer.buffer) > 0:
        metrics = ppo_trainer.update(last_value=0.0)
        ppo_losses.append(metrics.get('total_loss', 0.0))
        ppo_values.append(metrics.get('value_loss', 0.0))
        ppo_trainer.save_checkpoint(str(PPO_PATH), extra={'game': ppo_trainer.total_steps})

    if game_idx % 10 == 0:
        w20 = game_rewards[-20:]
        winner = min(result.placements, key=result.placements.get)
        print(f'Game {game_idx:3d}/{N_GAMES} | reward={mean_r:+.3f} '
              f'(avg20={np.mean(w20):+.3f}) | winner=P{winner} | '
              f'steps={ppo_trainer.total_steps} updates={ppo_trainer.update_count}')

print(f'Done — steps={ppo_trainer.total_steps}, updates={ppo_trainer.update_count}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(game_rewards, alpha=0.3, color='steelblue')
if len(game_rewards) >= 10:
    w = 10
    smooth = np.convolve(game_rewards, np.ones(w)/w, mode='valid')
    ax.plot(range(w-1, len(game_rewards)), smooth, color='steelblue', lw=2, label=f'{w}-game avg')
ax.axhline(0, color='gray', lw=0.8, ls='--')
ax.set_xlabel('Game'); ax.set_ylabel('Mean reward')
ax.set_title('Training Reward'); ax.legend()

ax = axes[1]
if ppo_losses:
    ax.plot(ppo_losses, marker='o', ms=4, color='crimson', label='total loss')
    ax.plot(ppo_values, marker='s', ms=4, color='orange',  label='value loss')
    ax.set_xlabel('PPO update'); ax.set_ylabel('Loss')
    ax.set_title('PPO Losses'); ax.legend()
else:
    ax.text(0.5, 0.5, 'No updates yet\n(run training cell first)',
            ha='center', va='center', transform=ax.transAxes)
    ax.set_title('PPO Losses')

plt.tight_layout()
plt.show()

---
## Example Turn Inspection

Run one game with a **deterministic (greedy)** policy and record every decision P0 makes.
We then inspect the action sequence, value estimates, and action probabilities to understand
what strategy the agent has learned.

In [ ]:
class RecordingAgent(_Agent):
    """Greedy agent that logs every decision for post-game analysis."""
    def __init__(self, policy, pid, device='cpu'):
        class _NullTrainer:
            def collect_transition(self, **_): pass
        super().__init__(policy, _NullTrainer(), pid, device)
        self.log = []

    def get_action(self, obs):
        ps  = obs['player_state']
        dev = torch.device(self.device)
        # Cache masks before state is mutated
        self._cached_type_mask = build_type_mask(ps).numpy()
        t_mask = torch.from_numpy(self._cached_type_mask).unsqueeze(0).to(dev)
        p_mask = build_pointer_mask(ps, -1).unsqueeze(0).to(dev)
        bt = torch.tensor(obs['board_tokens'][None], dtype=torch.float32, device=dev)
        st = torch.tensor(obs['shop_tokens'][None],  dtype=torch.float32, device=dev)
        ht = torch.tensor(obs['hand_tokens'][None],  dtype=torch.float32, device=dev)
        sc = torch.tensor(obs['scalar_context'][None], dtype=torch.float32, device=dev)
        op = obs.get('opp_tokens')
        ot = torch.tensor(op[None], dtype=torch.float32, device=dev) if op is not None else None

        with torch.no_grad():
            type_idx, ptr_idx, _, value = self.policy.get_action(
                bt, st, ht, sc, type_mask=t_mask, pointer_mask=p_mask,
                deterministic=True, opp_tokens=ot,
            )
            self._cached_ptr_mask = build_pointer_mask(ps, type_idx).numpy()
            out = self.policy(bt, st, ht, sc, opp_tokens=ot)
            type_probs = torch.softmax(out[0][0], dim=-1).cpu().numpy()

        self.log.append({
            'round':       ps.round_num,
            'gold':        ps.gold,
            'tavern_tier': ps.tavern_tier,
            'board':       [m.name for m in ps.board],
            'hand':        [m.name for m in ps.hand],
            'shop':        [m.name for m in ps.shop if m is not None],
            'action_type': type_idx,
            'action_name': ACTION_NAMES[type_idx],
            'ptr':         ptr_idx,
            'value_est':   float(value),
            'type_probs':  type_probs.tolist(),
        })
        return type_idx, ptr_idx

print('RecordingAgent ready.')

In [ ]:
TRACE_SEED = 42
policy_train.eval()

rec_agent = RecordingAgent(policy_train, pid=0, device=DEVICE)
agents_trace = [rec_agent] + [
    _Agent(policy_train, ppo_trainer, pid, DEVICE) for pid in range(1, N_PLAYERS)
]

game_trace = BattlegroundsGame(
    card_defs=card_defs_train, agents=agents_trace,
    board_computer=SymbolicBoardComputer(card_defs_train),
    firestone_client=FirestoneClient(firestone_path=None, mock_mode=True),
    matchmaker=Matchmaker(n_players=N_PLAYERS, seed=TRACE_SEED),
    tavern_pool=TavernPool(card_defs_train, seed=TRACE_SEED),
    n_players=N_PLAYERS, seed=TRACE_SEED,
)
result_trace = game_trace.run_game()

placement = result_trace.placements.get(0, '?')
print(f'P0 finished {placement}/8  |  {result_trace.n_rounds} rounds  |  {len(rec_agent.log)} action steps logged')

In [ ]:
PTR_SHOP_OFF  = 7
PTR_BOARD_OFF = 0
PTR_HAND_OFF  = 14

def resolve_target(a):
    typ, ptr = a['action_name'], a['ptr']
    if typ == 'BUY':
        idx = ptr - PTR_SHOP_OFF
        return a['shop'][idx] if 0 <= idx < len(a['shop']) else ''
    elif typ == 'SELL':
        return a['board'][ptr] if 0 <= ptr < len(a['board']) else ''
    elif typ == 'PLACE':
        idx = ptr - PTR_HAND_OFF
        return a['hand'][idx] if 0 <= idx < len(a['hand']) else ''
    return ''

for round_num, grp in groupby(rec_agent.log, key=lambda x: x['round']):
    actions = list(grp)
    first = actions[0]
    print(f"{'='*65}")
    print(f"Round {round_num}  |  Tier {first['tavern_tier']}  |  Gold {first['gold']}")
    print(f"  Shop : {first['shop']}")
    print(f"  Board: {first['board'] or ['(empty)']}")
    print()
    for i, a in enumerate(actions):
        tgt      = resolve_target(a)
        tgt_str  = f'  {tgt}' if tgt else ''
        top_probs = sorted(enumerate(a['type_probs']), key=lambda x: -x[1])[:3]
        prob_str  = '  '.join(f"{ACTION_NAMES[j]}:{p:.2f}" for j, p in top_probs)
        print(f"  [{i+1:2d}] {a['action_name']:10s}{tgt_str:28s}  V={a['value_est']:+.2f}  top=[{prob_str}]")
    print(f"  Board end: {actions[-1]['board'] or ['(empty)']}")
print('='*65)

In [ ]:
rounds_seen = sorted(set(a['round'] for a in rec_agent.log))

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=False)

# Value estimate over all steps
ax = axes[0]
values = [a['value_est'] for a in rec_agent.log]
ax.plot(values, color='steelblue', lw=1.5, alpha=0.85)
ax.axhline(0, color='gray', lw=0.8, ls='--')
cum = 0
for rn, grp in groupby(rec_agent.log, key=lambda x: x['round']):
    cnt = len(list(grp))
    ax.axvline(cum, color='orange', lw=0.7, alpha=0.5)
    ax.text(cum + 0.2, ax.get_ylim()[1]*0.85 if ax.get_ylim()[1] else 0.5,
            f'R{rn}', fontsize=7, color='darkorange')
    cum += cnt
ax.set_title('Value Estimate V(s) — P0 Throughout the Game')
ax.set_ylabel('V(s)'); ax.set_xlabel('Action step')

# Action-type distribution per round (stacked bar)
ax = axes[1]
type_counts = {n: [] for n in ACTION_NAMES}
for rn in rounds_seen:
    acts = [a['action_name'] for a in rec_agent.log if a['round'] == rn]
    for n in ACTION_NAMES:
        type_counts[n].append(acts.count(n))

x = np.arange(len(rounds_seen))
bottom = np.zeros(len(rounds_seen))
palette = ['#2196F3','#F44336','#4CAF50','#FF9800','#9C27B0','#00BCD4','#FFEB3B','#607D8B']
for name, color in zip(ACTION_NAMES, palette):
    counts = np.array(type_counts[name], dtype=float)
    if counts.any():
        ax.bar(x, counts, bottom=bottom, label=name, color=color, alpha=0.85)
        bottom += counts
ax.set_xticks(x)
ax.set_xticklabels([f'R{r}' for r in rounds_seen])
ax.set_ylabel('# actions')
ax.set_title('Action Type Distribution per Round  (P0)')
ax.legend(ncol=4, fontsize=8, loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
# Deep-dive into one round: action-prob heatmap + value trace
INSPECT_ROUND = min(5, max(rounds_seen))  # change to any round you like

ra = [a for a in rec_agent.log if a['round'] == INSPECT_ROUND]
if not ra:
    print(f'Round {INSPECT_ROUND} not in log — available: {rounds_seen}')
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, max(3, len(ra) * 0.45 + 2)))

    ax = axes[0]
    prob_matrix = np.array([a['type_probs'] for a in ra])
    im = ax.imshow(prob_matrix, aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)
    ax.set_xticks(range(len(ACTION_NAMES)))
    ax.set_xticklabels(ACTION_NAMES, rotation=40, ha='right', fontsize=9)
    ax.set_yticks(range(len(ra)))
    ylabels = [f"[{i+1}] {a['action_name']}" + (f' {resolve_target(a)[:15]}' if resolve_target(a) else '')
               for i, a in enumerate(ra)]
    ax.set_yticklabels(ylabels, fontsize=8)
    ax.set_title(f'Action Probabilities — Round {INSPECT_ROUND}')
    plt.colorbar(im, ax=ax, shrink=0.8)

    ax = axes[1]
    vals = [a['value_est'] for a in ra]
    ax.plot(vals, marker='o', ms=5, color='steelblue', lw=1.5)
    ax.axhline(0, color='gray', lw=0.8, ls='--')
    ax.set_xticks(range(len(ra)))
    ax.set_xticklabels([a['action_name'] for a in ra], rotation=40, ha='right', fontsize=8)
    ax.set_ylabel('V(s)')
    ax.set_title(f'Value Estimate — Round {INSPECT_ROUND}')

    plt.tight_layout()
    plt.show()

    print(f'\nRound {INSPECT_ROUND} | Tier {ra[0]["tavern_tier"]} | Gold {ra[0]["gold"]}')
    print(f'Shop:       {ra[0]["shop"]}')
    print(f'Board start:{ra[0]["board"] or ["(empty)"]}')
    for i, a in enumerate(ra):
        tgt = resolve_target(a)
        print(f'  [{i+1:2d}] {a["action_name"]:10s}  {tgt}')
    print(f'Board end:  {ra[-1]["board"] or ["(empty)"]}')